## Gold — City Comparison

Aggregates KPIs from all gold marts into a single long-format table for cross-city comparison.
One row per metric per city — easy to pivot for dashboards.

### Source
- `gtfs_gold.kpi_network` — network KPIs per city and transport type
- `gtfs_gold.hourly_coverage` — hourly coverage per city and day type
- `gtfs_gold.od_flows` — OD flows per city

### Output
`gtfs_gold.city_comparison`

| Column | Description |
|--------|-------------|
| city | City name |
| metric_name | Name of the metric |
| metric_value | Numeric value of the metric |
| metric_unit | Unit of measure (count, minutes) |

### Logic
Uses `UNION ALL` to stack metrics from different gold marts into a single long-format table.
Final pivot query reshapes the table with cities as columns for visual comparison.

In [0]:
SHOW COLUMNS IN gtfs_gold.hourly_coverage

In [0]:
CREATE OR REPLACE TABLE gtfs_gold.city_comparison AS
WITH kpi_pivot AS(
    SELECT
        city,
        'total_routes' AS metric_name,
        CAST(SUM(total_routes) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.kpi_network
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'total_trips_weekday' AS metric_name,
        CAST(SUM(total_trips_weekday) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.kpi_network
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'total_trips_weekend' AS metric_name,
        CAST(SUM(total_trips_weekend) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.kpi_network
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'avg_stops_per_route' AS metric_name,
        CAST(AVG(avg_stops_per_route) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.kpi_network
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'total_stops_served' AS metric_name,
        CAST(SUM(total_stops_served) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.kpi_network
    GROUP BY city
),
hourly_coverage_pivot AS(
    SELECT 
        city,
        'max_active_trips_weekday' AS metric_name, 
        CAST(MAX(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_trips_weekday' AS metric_name, 
        CAST(MIN(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_trips_weekday' AS metric_name, 
        CAST(AVG(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'max_active_trips_weekend' AS metric_name, 
        CAST(MAX(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_trips_weekend' AS metric_name, 
        CAST(MIN(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_trips_weekend' AS metric_name, 
        CAST(AVG(active_trips) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'max_active_stops_weekday' AS metric_name, 
        CAST(MAX(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_stops_weekday' AS metric_name, 
        CAST(MIN(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_stops_weekday' AS metric_name, 
        CAST(AVG(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday'
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'max_active_stops_weekend' AS metric_name, 
        CAST(MAX(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday')
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_stops_weekend' AS metric_name, 
        CAST(MIN(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_stops_weekend' AS metric_name, 
        CAST(AVG(active_stops) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'max_active_routes_weekday' AS metric_name, 
        CAST(MAX(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_routes_weekday' AS metric_name, 
        CAST(MIN(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_routes_weekday' AS metric_name, 
        CAST(AVG(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type = 'Weekday' 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'max_active_routes_weekend' AS metric_name, 
        CAST(MAX(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'min_active_routes_weekend' AS metric_name, 
        CAST(MIN(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
    UNION ALL
    SELECT 
        city, 
        'avg_active_routes_weekend' AS metric_name, 
        CAST(AVG(active_routes) AS DOUBLE) AS metric_value, 
        'count' AS metric_unit
    FROM gtfs_gold.hourly_coverage 
    WHERE day_type IN ('Saturday','Sunday') 
    GROUP BY city
),
od_flows_pivot AS(
    SELECT
        city,
        'avg_travel_time_minutes' AS metric_name,
        CAST(AVG(avg_travel_time_minutes) AS DOUBLE) AS metric_value,
        'minutes' AS metric_unit
    FROM gtfs_gold.od_flows
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'max_travel_time_minutes' AS metric_name,
        CAST(MAX(avg_travel_time_minutes) AS DOUBLE) AS metric_value,
        'minutes' AS metric_unit
    FROM gtfs_gold.od_flows
    GROUP BY city
    UNION ALL
    SELECT
        city,
        'min_travel_time_minutes' AS metric_name,
        CAST(MIN(avg_travel_time_minutes) AS DOUBLE) AS metric_value,
        'minutes' AS metric_unit
    FROM gtfs_gold.od_flows
    GROUP BY city
    UNION ALL    
    SELECT
        city,
        'total_od_couple' AS metric_name,
        CAST(COUNT(*) AS DOUBLE) AS metric_value,
        'count' AS metric_unit
    FROM gtfs_gold.od_flows
    GROUP BY city
)
SELECT * FROM kpi_pivot
UNION ALL
SELECT * FROM hourly_coverage_pivot
UNION ALL
SELECT * FROM od_flows_pivot

In [0]:
SELECT city, metric_name, ROUND(metric_value, 2) as metric_value, metric_unit
FROM gtfs_gold.city_comparison
ORDER BY metric_name, city

In [0]:
SELECT
    metric_name,
    metric_unit,
    MAX(CASE WHEN city = 'roma'   THEN ROUND(metric_value, 2) END) AS roma,
    MAX(CASE WHEN city = 'milano' THEN ROUND(metric_value, 2) END) AS milano,
    MAX(CASE WHEN city = 'torino' THEN ROUND(metric_value, 2) END) AS torino
FROM gtfs_gold.city_comparison
GROUP BY metric_name, metric_unit
ORDER BY metric_name